# 05 — 3D SchNet + Boltzmann-Informed Conformer Aggregation

`conformers (notebook 02) -> SchNet per-conformer embedding -> Boltzmann aggregation -> classifier`.

This notebook trains three aggregation variants for a direct, controlled comparison:
- **uniform_mean** — plain average across conformers, ignores energies entirely (control)
- **weighted_mean** — direct Boltzmann-weighted average (baseline, satisfies the core requirement)
- **learned_attention** — learned attention logits + Boltzmann log-weights as a prior bias (the
  "learnable attention mechanism informed by Boltzmann probabilities" variant)


In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from pathlib import Path

import yaml
import torch
from torch.utils.data import DataLoader

from src.utils import load_json, save_json, set_seed
from src.datasets import ConformerDataset, collate_conformers
from src.models import Conformer3DModel
from src.train_utils import train_binary_classifier, evaluate

with open("../config/model_3d.yaml") as f:
    cfg = yaml.safe_load(f)

set_seed(cfg["train"]["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## Load conformer-backed datasets

Reads the `.npz` files cached in notebook 02. Uses the same `splits.json` (already filtered for the 4 ETKDG failures), so ids here are guaranteed to have a matching conformer file.

In [2]:
splits = load_json(cfg["paths"]["splits_file"])

conf_datasets = {
    split_name: ConformerDataset(
        splits[split_name]["id"], splits[split_name]["y"], cfg["paths"]["conformers_dir"]
    )
    for split_name in ["train", "valid", "test"]
}
print({k: len(v) for k, v in conf_datasets.items()})

# sanity check the batch shape before training
sample_loader = DataLoader(conf_datasets["train"], batch_size=4, collate_fn=collate_conformers)
sample_batch = next(iter(sample_loader))
{k: (v.shape if hasattr(v, 'shape') else v) for k, v in sample_batch.items()}


{'train': 453, 'valid': 66, 'test': 132}


{'atom_z': torch.Size([1171]),
 'atom_pos': torch.Size([1171, 3]),
 'atom_conf_batch': torch.Size([1171]),
 'conf_mol_batch': torch.Size([21]),
 'conf_weights': torch.Size([21]),
 'label': torch.Size([4]),
 'num_mols': 4}

## Shared training helper for all three aggregation modes

Same `train_binary_classifier` as every other notebook — only `agg_mode` changes.

In [3]:
def train_3d_variant(agg_mode, experiment_dir):
    loaders = {
        split_name: DataLoader(
            conf_datasets[split_name], batch_size=cfg["train"]["batch_size"],
            shuffle=(split_name == "train"), collate_fn=collate_conformers,
        )
        for split_name in ["train", "valid", "test"]
    }

    model = Conformer3DModel(
        agg_mode=agg_mode,
        hidden_channels=cfg["model"]["hidden_channels"], num_filters=cfg["model"]["num_filters"],
        num_interactions=cfg["model"]["num_interactions"], num_gaussians=cfg["model"]["num_gaussians"],
        cutoff=cfg["model"]["cutoff"], max_z=cfg["model"]["max_z"],
    )
    print(f"[{agg_mode}] params: {sum(p.numel() for p in model.parameters()):,}")

    def forward_fn(model, batch, device):
        return model(
            batch["atom_z"].to(device), batch["atom_pos"].to(device),
            batch["atom_conf_batch"].to(device), batch["conf_mol_batch"].to(device),
            batch["conf_weights"].to(device), num_mols=batch["num_mols"],
        )

    model, train_info = train_binary_classifier(
        model, loaders["train"], loaders["valid"], forward_fn, cfg, experiment_dir, device
    )
    val_metrics = evaluate(model, loaders["valid"], forward_fn, device)
    print(f"[{agg_mode}] val AUROC: {val_metrics['auroc']:.4f}")

    save_json(
        {"model": f"3d_{agg_mode}", **val_metrics, **train_info, "config": cfg},
        Path(experiment_dir) / "metrics.json",
    )
    return val_metrics["auroc"]


## Main run: weighted_mean (this is the 3D-only ablation row, and feeds notebook 06)

In [4]:
auroc_weighted_mean = train_3d_variant("weighted_mean", cfg["paths"]["experiment_dir"])


[weighted_mean] params: 238,465
epoch   1  train_loss=1.5351  val_loss=0.5391  val_auroc=0.7100
epoch   2  train_loss=0.6693  val_loss=0.6029  val_auroc=0.6725
epoch   3  train_loss=0.6904  val_loss=0.7398  val_auroc=0.6800
epoch   4  train_loss=0.6508  val_loss=0.9019  val_auroc=0.7094
epoch   5  train_loss=0.5522  val_loss=0.6138  val_auroc=0.7144
epoch   6  train_loss=0.5588  val_loss=0.6754  val_auroc=0.6469
epoch   7  train_loss=0.6003  val_loss=0.6601  val_auroc=0.6794
epoch   8  train_loss=0.5177  val_loss=0.5951  val_auroc=0.6625
epoch   9  train_loss=0.4935  val_loss=0.7071  val_auroc=0.6575
epoch  10  train_loss=0.4916  val_loss=0.6116  val_auroc=0.7325
epoch  11  train_loss=0.4727  val_loss=0.6900  val_auroc=0.6825
epoch  12  train_loss=0.4850  val_loss=0.5623  val_auroc=0.7081
epoch  13  train_loss=0.4812  val_loss=0.4962  val_auroc=0.7656
epoch  14  train_loss=0.4632  val_loss=0.4600  val_auroc=0.7837
epoch  15  train_loss=0.4365  val_loss=0.5329  val_auroc=0.7563
epoch  1

## Boltzmann-weighting ablation: uniform_mean vs. learned_attention

Same architecture, same training procedure — only the aggregation changes. This is the direct evidence for whether Boltzmann weighting (vs. ignoring energies) actually helps.

In [5]:
auroc_uniform_mean = train_3d_variant("uniform_mean", "../experiments/3d_uniform_mean")


[uniform_mean] params: 238,465
epoch   1  train_loss=1.3988  val_loss=1.0418  val_auroc=0.5012
epoch   2  train_loss=0.8037  val_loss=0.6119  val_auroc=0.6381
epoch   3  train_loss=0.6403  val_loss=0.5633  val_auroc=0.6856
epoch   4  train_loss=0.5614  val_loss=0.5567  val_auroc=0.7331
epoch   5  train_loss=0.5812  val_loss=0.5651  val_auroc=0.6975
epoch   6  train_loss=0.4918  val_loss=0.5437  val_auroc=0.7400
epoch   7  train_loss=0.4692  val_loss=0.5556  val_auroc=0.7363
epoch   8  train_loss=0.5003  val_loss=0.5775  val_auroc=0.7050
epoch   9  train_loss=0.5602  val_loss=0.7326  val_auroc=0.6581
epoch  10  train_loss=0.5564  val_loss=0.5590  val_auroc=0.6413
epoch  11  train_loss=0.5360  val_loss=0.7389  val_auroc=0.5913
epoch  12  train_loss=0.4913  val_loss=0.5282  val_auroc=0.7087
epoch  13  train_loss=0.4819  val_loss=0.5792  val_auroc=0.6994
epoch  14  train_loss=0.4592  val_loss=0.4915  val_auroc=0.7375
epoch  15  train_loss=0.4535  val_loss=0.5820  val_auroc=0.7213
epoch  16

In [6]:
auroc_learned_attention = train_3d_variant("learned_attention", "../experiments/3d_learned_attention")


[learned_attention] params: 246,786
epoch   1  train_loss=1.8044  val_loss=0.5524  val_auroc=0.7063
epoch   2  train_loss=0.9653  val_loss=0.6674  val_auroc=0.6312
epoch   3  train_loss=0.8427  val_loss=0.5654  val_auroc=0.7338
epoch   4  train_loss=0.6695  val_loss=0.5306  val_auroc=0.6963
epoch   5  train_loss=0.5969  val_loss=0.8173  val_auroc=0.6381
epoch   6  train_loss=0.5858  val_loss=0.5553  val_auroc=0.7288
epoch   7  train_loss=0.6088  val_loss=0.5861  val_auroc=0.6837
epoch   8  train_loss=0.5446  val_loss=0.6442  val_auroc=0.6775
epoch   9  train_loss=0.4851  val_loss=0.6641  val_auroc=0.7669
epoch  10  train_loss=0.4746  val_loss=0.5801  val_auroc=0.6950
epoch  11  train_loss=0.4553  val_loss=0.6774  val_auroc=0.7038
epoch  12  train_loss=0.4930  val_loss=0.5726  val_auroc=0.7388
epoch  13  train_loss=0.4554  val_loss=0.6596  val_auroc=0.6869
epoch  14  train_loss=0.4527  val_loss=0.5351  val_auroc=0.7481
epoch  15  train_loss=0.5458  val_loss=0.4860  val_auroc=0.7638
epoc

## Compare all three

In [7]:
import pandas as pd

comparison = pd.DataFrame({
    "aggregation": ["uniform_mean (control)", "weighted_mean (Boltzmann)", "learned_attention (Boltzmann prior)"],
    "val_auroc": [auroc_uniform_mean, auroc_weighted_mean, auroc_learned_attention],
})
comparison


,aggregation,val_auroc
0,uniform_mean (control),0.848125
1,weighted_mean (Boltzmann),0.825000
2,learned_attention (Boltzmann prior),0.886250


## Compare against 1D / 2D so far

In [9]:
d1 = load_json("../experiments/1d_only/metrics.json")
d2 = load_json("../experiments/2d_only/metrics.json")
print(f"1D-only val AUROC:              {d1['auroc']:.4f}")
print(f"2D-only val AUROC:              {d2['auroc']:.4f}")
print(f"3D-only (learned attention) AUROC:  {auroc_learned_attention:.4f}")


1D-only val AUROC:              0.8462
2D-only val AUROC:              0.8812
3D-only (learned attention) AUROC:  0.8862
